# Run on Colab

Trains and/or evaluates one run. Every step only calls code from the repo.

Use a GPU runtime (Runtime > Change runtime type > T4 GPU), set the parameters below, then Runtime > Run all. Running it again resumes: checkpoints are on Drive, and a finished run is skipped.

Everything is kept under `MyDrive/cifar10-robustness/`: `runs/` (run folders), `data/` (CIFAR-10-C and the CIFAR-10 archive), `results/` (result files and the console logs).

In [ ]:
MODEL = "cnn"  # cnn | resnet18
STRATEGY = "standard"  # standard | augmix | corruption_aware | none
SEED = 0  # 0 | 1 | 2
STAGE = "both"  # train | evaluate | both
RUN_TIMING = False  # True: after the run, time 2 epochs of the Phase 07 timing configs

REPO_URL = "https://github.com/aysha0559/Image-Classification-with-ResNet-18.git"
BRANCH = "phase-07-pilot"
REPO_DIR = "/content/cifar10-robustness-study"

In [ ]:
import os
import shutil
import time

DRIVE_ROOT = "/content/drive/MyDrive/cifar10-robustness"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results"
LOCAL_DATA = "/content/data"
RUNS_DIR = f"{DRIVE_ROOT}/runs"
CIFAR10_ARCHIVE = "cifar-10-python.tar.gz"

RUN_ID = f"{MODEL}-{STRATEGY}-s{SEED}"
CONFIG = f"configs/{MODEL}_{STRATEGY}.yaml"
RUN_DIR = f"{RUNS_DIR}/{RUN_ID}"
TEE = f"2>&1 | tee -a {DRIVE_RESULTS}/logs/{RUN_ID}.console.log"
TIMING_TEE = f"2>&1 | tee -a {DRIVE_RESULTS}/logs/timing.console.log"
KEEP = f"{DRIVE_RESULTS}/run_folders"

os.environ["DATA_DIR"] = LOCAL_DATA
os.environ["RUNS_DIR"] = RUNS_DIR
os.environ["PYTHONUNBUFFERED"] = "1"


def require_ok(step):
    """Stop Run all if the last `!` command failed."""
    assert get_ipython().user_ns["_exit_code"] == 0, f"{step} failed; read the output above"

## Setup

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
for folder in (LOCAL_DATA, DRIVE_DATA, RUNS_DIR, f"{DRIVE_RESULTS}/logs"):
    os.makedirs(folder, exist_ok=True)

In [ ]:
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
require_ok("getting the repo")
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt
require_ok("pip install")

Stops here if there is no GPU: `device: auto` would otherwise train on the CPU without a word.

In [ ]:
!python -V
!python -c "import torch, torchvision as tv; print(torch.__version__, tv.__version__)"
!python -c "import torch; print(torch.cuda.get_device_name(0))"
require_ok("the GPU check (Runtime > Change runtime type > T4 GPU)")

In [ ]:
!mkdir -p results/raw
!cp -n {DRIVE_RESULTS}/raw/* results/raw/ 2>/dev/null
!bash -c "echo === session $(date -u +%FT%TZ) commit $(git rev-parse HEAD) {TEE}"

## Data

First session: downloads CIFAR-10 and CIFAR-10-C (about 2.9 GB) to the fast local disk, verifies them, keeps a copy on Drive and waits until Drive has it all. Do not disconnect before that finishes.

Later sessions: copies both from Drive and re-checks every file against `splits/cifar10c_checksums.json`, with no download. If that check fails, a first session was cut short: delete `MyDrive/cifar10-robustness/data/CIFAR-10-C` and run again.

In [ ]:
DRIVE_ARCHIVE = f"{DRIVE_DATA}/{CIFAR10_ARCHIVE}"
LOCAL_ARCHIVE = f"{LOCAL_DATA}/{CIFAR10_ARCHIVE}"
DRIVE_C10C = f"{DRIVE_DATA}/CIFAR-10-C"
LOCAL_C10C = f"{LOCAL_DATA}/CIFAR-10-C"
first_session = not (os.path.exists(DRIVE_ARCHIVE) and os.path.isdir(DRIVE_C10C))

if os.path.exists(DRIVE_ARCHIVE):
    shutil.copy(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
!python -c "from src.data.cifar10 import get_test_dataset; get_test_dataset('{LOCAL_DATA}')"
require_ok("getting CIFAR-10")
if not os.path.exists(DRIVE_ARCHIVE):
    shutil.copy(LOCAL_ARCHIVE, DRIVE_ARCHIVE)

if os.path.isdir(DRIVE_C10C):
    if not os.path.isdir(LOCAL_C10C):
        shutil.copytree(DRIVE_C10C, LOCAL_C10C)
    !python -m scripts.download_cifar10c --data-dir {LOCAL_DATA} --verify-only
    require_ok("verifying CIFAR-10-C")
else:
    !python -m scripts.download_cifar10c --data-dir {LOCAL_DATA}
    require_ok("downloading CIFAR-10-C")
    shutil.copytree(LOCAL_C10C, DRIVE_C10C)
    os.remove(f"{LOCAL_DATA}/CIFAR-10-C.tar")

if first_session:
    drive.flush_and_unmount()
    drive.mount("/content/drive")

## Train

In [ ]:
if STAGE in ("train", "both"):
    !bash -o pipefail -c "python -m src.train --config {CONFIG} --seed {SEED} {TEE}"
    require_ok("training")
    !cat {RUN_DIR}/run_meta.json

## Evaluate

Skipped if this run's results are already saved on Drive.

In [ ]:
if STAGE in ("evaluate", "both"):
    if os.path.exists(f"results/raw/{RUN_ID}.csv"):
        print(f"{RUN_ID}: results already exist, not evaluating again")
    else:
        started = time.time()
        !bash -o pipefail -c "python -m src.evaluate --run-dir {RUN_DIR} {TEE}"
        require_ok("evaluation")
        seconds = round(time.time() - started)
        !bash -c "echo evaluate wall time {seconds} s {TEE}"

## Save results to Drive

In [ ]:
!python -m scripts.registry
require_ok("the registry")
!mkdir -p {DRIVE_RESULTS}/raw {KEEP}
!cp results/raw/{RUN_ID}* {DRIVE_RESULTS}/raw/
!cp results/runs.csv {DRIVE_RESULTS}/
!cd {RUNS_DIR} && cp --parents {RUN_ID}/run_meta.json {RUN_ID}/train_log.csv {KEEP}/
!ls -la {DRIVE_RESULTS}/raw

## Timing (Phase 07, only with `RUN_TIMING = True`)

Stops each config after 2 epochs. These `timing-*` runs never finish, so they never reach `results/`.

ResNet-18 evaluation time cannot come from them, because only a finished run can be evaluated. So a throwaway 1-epoch ResNet-18 on 128 images is evaluated on all 76 conditions; `--subset` makes it a `smoke-` run, and its numbers go to `results/smoke/`, not `results/raw/`.

In [ ]:
TIMING_CONFIGS = [
    "resnet18_standard",
    "resnet18_augmix",
    "resnet18_corruption_aware",
    "cnn_augmix",
    "cnn_corruption_aware",
]
EVAL_RUN = "smoke-timing-resnet18-eval"

if RUN_TIMING:
    for name in TIMING_CONFIGS:
        train = f"python -m src.train --config configs/{name}.yaml --seed 0"
        !bash -o pipefail -c "STOP_AFTER_EPOCHS=2 {train} --run-id timing-{name} {TIMING_TEE}"
        require_ok(f"timing {name}")

    train = "python -m src.train --config configs/resnet18_standard.yaml --seed 0"
    !bash -o pipefail -c "{train} --epochs 1 --subset 128 --run-id {EVAL_RUN} {TIMING_TEE}"
    require_ok("the ResNet-18 evaluation timing run")
    started = time.time()
    evaluate = f"python -m src.evaluate --run-dir {RUNS_DIR}/{EVAL_RUN} --force"
    !bash -o pipefail -c "{evaluate} {TIMING_TEE}"
    require_ok("the ResNet-18 evaluation timing")
    seconds = round(time.time() - started)
    !bash -c "echo resnet18 evaluate wall time {seconds} s {TIMING_TEE}"

    !mkdir -p {KEEP}
    !cd {RUNS_DIR} && cp --parents timing-*/run_meta.json timing-*/train_log.csv {KEEP}/

## Finish

Waits until everything has reached Drive, then unmounts it. After this it is safe to disconnect.

In [ ]:
drive.flush_and_unmount()
print("Drive is up to date.")